# 🧠 Scratchformer — Training Notebook (Day 5)

This notebook trains our from-scratch GPT model on a **Colab T4 GPU**.

**Workflow:**
1. Clone the latest code from GitHub
2. Install dependencies
3. Prepare data (download + tokenize Tiny Shakespeare)
4. Train the model with checkpoints saved to Google Drive
5. Plot loss curves and generate sample text

---

### ⚡ Before you start
1. **Runtime → Change runtime type → T4 GPU**
2. Run the cells in order
3. Checkpoints are saved to Google Drive so you don't lose them if the Colab session disconnects

## 1. Setup — Clone Repo & Install Dependencies

In [ ]:
# Clone the latest code from GitHub
# This ensures Colab always has the most recent version of your .py files
!rm -rf scratchformer
!git clone https://github.com/aryannten/scratchformer.git
%cd scratchformer
!pip install -q -r requirements.txt

In [ ]:
# Mount Google Drive for persistent checkpoint storage
# Colab VMs are ephemeral — if the session dies, local files are gone.
# Drive is your safety net.
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/scratchformer_checkpoints'
import os
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_CHECKPOINT_DIR}')

In [ ]:
# Verify GPU is available
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:             {torch.cuda.get_device_name(0)}')
    print(f'Memory:          {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected! Training will be very slow on CPU.')
    print('    Go to Runtime → Change runtime type → T4 GPU')

## 2. Prepare Data

In [ ]:
# Download and tokenize the Tiny Shakespeare dataset
# This creates:
#   data/prepared/train.pt  — 90% of the data, tokenized as integers
#   data/prepared/val.pt    — 10% held out for evaluation
#   data/prepared/vocab.json — the character vocabulary
!python prepare_data.py --dataset shakespeare

In [ ]:
# Quick sanity check: load the data and inspect
from tokenizer import CharTokenizer

tokenizer = CharTokenizer.load('data/prepared/vocab.json')
train_data = torch.load('data/prepared/train.pt', weights_only=True)
val_data = torch.load('data/prepared/val.pt', weights_only=True)

print(f'Vocab size:    {tokenizer.vocab_size} characters')
print(f'Train tokens:  {len(train_data):,}')
print(f'Val tokens:    {len(val_data):,}')
print(f'\nSample text (first 200 chars):')
print(tokenizer.decode(train_data[:200].tolist()))

## 3. Pre-Training Sanity Check

Before spending GPU time on a full run, let's verify the model works:
- Output shape is correct
- Initial loss is ~4.17 (= ln(65), random guessing among 65 characters)
- Loss decreases after a few gradient steps

In [ ]:
from model import Scratchformer, GPTConfig
from train import get_batch
import math

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Create model with default config
config = GPTConfig(vocab_size=tokenizer.vocab_size)
model = Scratchformer(config).to(device)

print(f'Model parameters: {model.count_parameters():,}')
print(f'Expected initial loss: {math.log(tokenizer.vocab_size):.4f}')

# Test forward pass
x, y = get_batch(train_data, batch_size=4, block_size=config.block_size, device=device)
logits, loss = model(x, y)

print(f'Output shape:    {logits.shape}  (expected: [4, {config.block_size}, {tokenizer.vocab_size}])')
print(f'Initial loss:    {loss.item():.4f}')
print(f'Loss is sane:    {"✅ Yes" if abs(loss.item() - math.log(tokenizer.vocab_size)) < 0.5 else "❌ No — investigate!"}')

In [ ]:
# Quick gradient step test: does loss actually decrease?
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

losses_check = []
for i in range(20):
    x, y = get_batch(train_data, batch_size=32, block_size=config.block_size, device=device)
    _, loss = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses_check.append(loss.item())

print(f'Loss after  0 steps: {losses_check[0]:.4f}')
print(f'Loss after 20 steps: {losses_check[-1]:.4f}')
print(f'Loss decreased: {"✅ Yes — pipeline works!" if losses_check[-1] < losses_check[0] else "❌ No — something is wrong"}')

# Clean up — we'll create a fresh model for the real training
del model, optimizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 4. Train! 🚀

This is the main training run. On a T4 GPU, 5000 steps with batch_size=64 should take **~5–15 minutes**.

Checkpoints are saved:
- Every 500 steps → `checkpoints/step_XXXX.pt`
- Best validation loss → `checkpoints/best.pt`
- After training → `checkpoints/final.pt`

In [ ]:
from train import train, TrainConfig
from model import GPTConfig

# ── Model architecture ────────────────────────────────────
model_config = GPTConfig(
    # vocab_size is set automatically from the tokenizer
    block_size = 128,     # context window: 128 characters
    n_layer    = 4,       # 4 transformer blocks
    n_head     = 4,       # 4 attention heads
    n_embd     = 128,     # 128-dimensional embeddings
)

# ── Training hyperparameters ──────────────────────────────
train_config = TrainConfig(
    dataset        = 'shakespeare',
    max_steps      = 5000,       # total training steps
    batch_size     = 64,         # sequences per batch
    learning_rate  = 3e-4,       # AdamW LR
    weight_decay   = 0.1,        # regularization
    grad_clip      = 1.0,        # gradient clipping
    warmup_steps   = 200,        # LR warmup
    eval_interval  = 250,        # eval every N steps
    save_interval  = 500,        # checkpoint every N steps
    checkpoint_dir = 'checkpoints',
)

# ── Launch training ───────────────────────────────────────
model, loss_log, tokenizer = train(
    model_config=model_config,
    train_config=train_config,
)

## 5. Results — Loss Curve

In [ ]:
# Display the loss curve
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Re-plot inline for the notebook
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

steps = [e['step'] for e in loss_log]
train_losses = [e['train'] for e in loss_log]
val_losses = [e['val'] for e in loss_log]

ax.plot(steps, train_losses, label='Train Loss', color='#4ECDC4', linewidth=2.5)
ax.plot(steps, val_losses, label='Val Loss', color='#FF6B6B', linewidth=2.5)
ax.set_xlabel('Step', fontsize=13)
ax.set_ylabel('Loss', fontsize=13)
ax.set_title('Scratchformer Training — Tiny Shakespeare', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

# Annotate start and end
ax.annotate(f'Start: {train_losses[0]:.2f}', xy=(steps[0], train_losses[0]),
            fontsize=10, color='#4ECDC4', fontweight='bold',
            xytext=(steps[0]+200, train_losses[0]+0.1),
            arrowprops=dict(arrowstyle='->', color='#4ECDC4'))
ax.annotate(f'Final: {val_losses[-1]:.2f}', xy=(steps[-1], val_losses[-1]),
            fontsize=10, color='#FF6B6B', fontweight='bold',
            xytext=(steps[-1]-800, val_losses[-1]+0.2),
            arrowprops=dict(arrowstyle='->', color='#FF6B6B'))

plt.tight_layout()
plt.show()

print(f'\nFinal train loss: {train_losses[-1]:.4f}')
print(f'Final val loss:   {val_losses[-1]:.4f}')

## 6. Generate Sample Text

Let's see what our trained model can produce! We'll try different sampling strategies.

In [ ]:
def generate_text(model, tokenizer, prompt='', max_tokens=300, temperature=0.8, top_k=40):
    """Generate text from the trained model."""
    device = next(model.parameters()).device
    model.eval()

    if prompt:
        tokens = tokenizer.encode(prompt)
        idx = torch.tensor([tokens], dtype=torch.long, device=device)
    else:
        # Start with a newline character (common start in Shakespeare)
        idx = torch.zeros((1, 1), dtype=torch.long, device=device)

    generated = model.generate(idx, max_new_tokens=max_tokens, temperature=temperature, top_k=top_k)
    return tokenizer.decode(generated[0].tolist())

In [ ]:
# Generation with different temperatures
print('=' * 60)
print('🎭 SAMPLE GENERATION — Temperature = 0.8 (balanced)')
print('=' * 60)
print(generate_text(model, tokenizer, temperature=0.8))

print('\n' + '=' * 60)
print('🧊 SAMPLE GENERATION — Temperature = 0.3 (conservative)')
print('=' * 60)
print(generate_text(model, tokenizer, temperature=0.3))

print('\n' + '=' * 60)
print('🔥 SAMPLE GENERATION — Temperature = 1.2 (creative)')
print('=' * 60)
print(generate_text(model, tokenizer, temperature=1.2))

In [ ]:
# Try prompting with a known Shakespeare-style opening
prompts = [
    'ROMEO:',
    'To be or not to be',
    'The king',
    'HAMLET:\nO,',
]

for prompt in prompts:
    print('=' * 60)
    print(f'📝 Prompt: "{prompt}"')
    print('=' * 60)
    output = generate_text(model, tokenizer, prompt=prompt, max_tokens=200, temperature=0.8)
    print(output)
    print()

## 7. Copy Checkpoints to Google Drive

Save the best and final checkpoints to Drive so they persist even after the Colab session ends.

In [ ]:
import shutil

# Copy key checkpoints to Google Drive
for ckpt_name in ['best.pt', 'final.pt', 'loss_curve.png']:
    src = f'checkpoints/{ckpt_name}'
    dst = f'{DRIVE_CHECKPOINT_DIR}/{ckpt_name}'
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'✅ Copied {ckpt_name} → {dst}')
    else:
        print(f'⚠️  {src} not found, skipping')

# Also save the vocab so we can decode later
shutil.copy2('data/prepared/vocab.json', f'{DRIVE_CHECKPOINT_DIR}/vocab.json')
print(f'✅ Copied vocab.json → {DRIVE_CHECKPOINT_DIR}/vocab.json')

print(f'\n📁 Drive contents:')
for f in os.listdir(DRIVE_CHECKPOINT_DIR):
    size = os.path.getsize(os.path.join(DRIVE_CHECKPOINT_DIR, f))
    print(f'   {f:30s} {size / 1e6:.1f} MB')

## 8. What to Look For

### Loss curve health check:
- ✅ **Both lines trend downward** — learning is happening
- ✅ **Train and val track each other** — not overfitting (yet)
- ⚠️ **Val loss flattens while train drops** — beginning to overfit (normal for small models on this dataset)
- ❌ **Loss is spiky or increasing** — LR too high, try reducing to 1e-4
- ❌ **Loss doesn't decrease at all** — bug in the code (but we verified it works in section 3!)

### Generation quality:
- At **~2000 steps**: gibberish that looks vaguely English-like
- At **~5000 steps**: words are mostly real, some coherent phrases
- The output should be clearly structured (dialogue format, line breaks) even if the content is nonsensical

### What a good final loss looks like:
- **Random init**: ~4.17 (ln(65))
- **After 5000 steps**: ~1.5–2.0 is typical for this model size
- **Below 1.5**: great — model is learning well

---

### ⏭️ Next: Day 6
If the loss curve looks healthy and generation produces semi-coherent Shakespeare, the pipeline is **verified end-to-end**. Day 6 will focus on the standalone `generate.py` with more sampling strategies.